# Layer V ET Identity Gene Analysis
Identifies genes that define Layer V ET identity through stepwise comparison:
- **Non-Neuron vs Layer V ET** → Neuron-identity genes
- **Inhibitory-Neuron vs Layer V ET** → Exitory-neuron-identity genes 
- **Upper Layer vs Layer V ET** → Deep-layer-identity genes  
- **Other Deep Layer vs Layer V ET** → Layer V ET-specific genes

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

sc.settings.verbosity = 1

/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: 

## 1. Load Data

In [2]:
# ── EDIT THIS PATH ──────────────────────────────────────────────────────────
H5AD_PATH = "/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad"
OUT_DIR   = Path("/home/nakagawa/datasets/LayerV_ET_results_rnaseq")
# ────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

AnnData object with n_obs × n_vars = 71183 × 30198
    obs: 'aggr_num', 'umi.counts', 'gene.counts', 'library_id', 'tube_barcode', 'Seq_batch', 'Region', 'Lib_type', 'donor_id', 'Amp_Name', 'Amp_Date', 'Amp_PCR_cyles', 'Lib_Date', 'Replicate_Lib', 'Lib_PCR_cycles', 'Lib_PassFail', 'Cell_Capture', 'Lib_Cells', 'Mean_Reads_perCell', 'Median_Genes_perCell', 'Median_UMI_perCell', 'Saturation', 'Live_percent', 'Total_Cells', 'Live_Cells', 'exp_component_name', 'mapped_reads', 'unmapped_reads', 'nonconf_mapped_reads', 'total.reads', 'doublet.score', 'row', 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'temp_class_label', 'BICCN_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'di

## 2. Find the Cell Type Column & List All Cell Types

In [19]:
# Show all metadata columns so you can identify the right cell type column
print("Available metadata columns:")
print(adata.obs.columns.tolist())
print(adata.obs['BICCN_cluster_label'].value_counts())

Available metadata columns:
['aggr_num', 'umi.counts', 'gene.counts', 'library_id', 'tube_barcode', 'Seq_batch', 'Region', 'Lib_type', 'donor_id', 'Amp_Name', 'Amp_Date', 'Amp_PCR_cyles', 'Lib_Date', 'Replicate_Lib', 'Lib_PCR_cycles', 'Lib_PassFail', 'Cell_Capture', 'Lib_Cells', 'Mean_Reads_perCell', 'Median_Genes_perCell', 'Median_UMI_perCell', 'Saturation', 'Live_percent', 'Total_Cells', 'Live_Cells', 'exp_component_name', 'mapped_reads', 'unmapped_reads', 'nonconf_mapped_reads', 'total.reads', 'doublet.score', 'row', 'BICCN_cluster_id', 'QC', 'BICCN_cluster_label', 'BICCN_subclass_label', 'BICCN_class_label', 'size', 'temp_class_label', 'BICCN_ontology_term_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_

In [20]:
# ── EDIT: set to the column name that contains cell type labels ──────────────
# Common names: 'cell_type', 'CellType', 'cluster', 'subclass_label', 'cell_type_alias_label'
CELLTYPE_COL = "BICCN_cluster_label"   # <-- change this if needed after checking output above
# ────────────────────────────────────────────────────────────────────────────

cell_types = sorted(adata.obs[CELLTYPE_COL].unique().tolist())
print(f"\nFound {len(cell_types)} cell types in '{CELLTYPE_COL}':\n")
for i, ct in enumerate(cell_types):
    n = (adata.obs[CELLTYPE_COL] == ct).sum()
    print(f"  [{i:02d}] {ct}  (n={n})")


Found 78 cell types in 'BICCN_cluster_label':

  [00] Astro Aqp4  (n=398)
  [01] Endo Slc38a5_1  (n=187)
  [02] L2/3 IT  (n=10915)
  [03] L5 ET_1  (n=65)
  [04] L5 ET_2  (n=39)
  [05] L5 ET_3  (n=57)
  [06] L5 IT Pld5  (n=653)
  [07] L5 IT S100b  (n=4272)
  [08] L5 IT Tcap_1  (n=7462)
  [09] L5 IT Tcap_2  (n=17334)
  [10] L5 NP Slc17a8_1  (n=633)
  [11] L5 NP Slc17a8_2  (n=495)
  [12] L5 NP Slc17a8_3  (n=131)
  [13] L6 CT Cpa6_1  (n=5854)
  [14] L6 CT Cpa6_2  (n=5986)
  [15] L6 CT Gpr139  (n=130)
  [16] L6 CT Nxph2 Kit  (n=34)
  [17] L6 CT Nxph2 Pou3f2_1  (n=268)
  [18] L6 CT Nxph2 Pou3f2_2  (n=535)
  [19] L6 IT Car3  (n=69)
  [20] L6 IT Sulf1_1  (n=245)
  [21] L6 IT Sulf1_2  (n=705)
  [22] L6 IT Sulf1_3  (n=1055)
  [23] L6 IT Sulf1_4  (n=2440)
  [24] L6 NP Trh_1  (n=1295)
  [25] L6 NP Trh_2  (n=167)
  [26] L6 NP Trh_3  (n=426)
  [27] L6b Kcnip1  (n=23)
  [28] L6b Ror1  (n=177)
  [29] L6b Rprm  (n=227)
  [30] L6b Shisa6  (n=127)
  [31] Lamp5 Egln3_2_1  (n=179)
  [32] Lamp5 Lhx6  (n=38

## 3. Define Layer V ET and Classify Other Cell Types
After seeing the list above, fill in the three categories below.

In [5]:
# ── EDIT THESE after reading cell type list above ────────────────────────────

LAYER_V_ET = "L5 ET"   # exact string from the list above

# Copy-paste cell type names from the printed list into each group
NON_NEURON_TYPES = [
    "Astro",       # Astrocyte
    "Endo",        # Endothelial
    "SMC",         # Smooth Muscle Cell
    "VLMC",        # Vascular Leptomeningeal Cell
]

UPPER_LAYER_TYPES = [
    "L2/3 IT",     # Classic upper layer
]

OTHER_DEEP_LAYER_TYPES = [
    "L5 IT",       # Deep but NOT ET
    "L5/6 NP",     # Near-projecting, deep layer
    "L6 CT",       # Corticothalamic
    "L6 IT",       # Deep IT
    "L6 IT Car3",  # Deep IT subtype
    "L6b",         # Subplate-like deep layer
]

# Your target
TARGET_TYPE = "L5 ET"

# Interneurons — consider excluding or treating separately
INTERNEURON_TYPES = [
    "Lamp5",
    "Pvalb",
    "Sncg",
    "Sst",
    "Vip",
]

# ── Thresholds ───────────────────────────────────────────────────────────────
LOG2FC_THRESH  = 1.0    # absolute log2 fold change cutoff (2-fold)
PVAL_THRESH    = 0.05   # adjusted p-value (FDR)
# ─────────────────────────────────────────────────────────────────────────────

print(f"Target: {LAYER_V_ET}")
print(f"Non-neuron types  : {NON_NEURON_TYPES}")
print(f"Upper layer types : {UPPER_LAYER_TYPES}")
print(f"Other deep layer  : {OTHER_DEEP_LAYER_TYPES}")
print(f"GABAergic inhibitory neurons : {INTERNEURON_TYPES}")

Target: L5 ET
Non-neuron types  : ['Astro', 'Endo', 'SMC', 'VLMC']
Upper layer types : ['L2/3 IT']
Other deep layer  : ['L5 IT', 'L5/6 NP', 'L6 CT', 'L6 IT', 'L6 IT Car3', 'L6b']
GABAergic inhibitory neurons : ['Lamp5', 'Pvalb', 'Sncg', 'Sst', 'Vip']


## 4. Preprocessing

In [6]:
# Use raw counts if available, otherwise use .X
if adata.raw is not None:
    print("Using adata.raw for DE analysis")
    adata_use = adata.raw.to_adata()
else:
    print("Using adata.X for DE analysis")
    adata_use = adata.copy()

# Copy cell type labels to adata_use
adata_use.obs[CELLTYPE_COL] = adata.obs[CELLTYPE_COL]

# Normalize if not already (check if values look like raw counts)
if adata_use.X.max() > 100:
    sc.pp.normalize_total(adata_use, target_sum=1e4)
    sc.pp.log1p(adata_use)
    print("Normalized and log1p transformed")
else:
    print("Data appears pre-normalized")

Using adata.raw for DE analysis
Normalized and log1p transformed


## 5. Run DE Analysis: Each Cell Type vs Layer V ET

In [7]:
def run_de(adata_use, celltype_col, group_a, group_b, log2fc_thresh, pval_thresh):
    """
    Run Wilcoxon rank-sum DE between group_a vs group_b.
    Positive log2FC = upregulated in group_a relative to group_b (Layer V ET).
    """
    mask = adata_use.obs[celltype_col].isin([group_a, group_b])
    sub  = adata_use[mask].copy()
    sub.obs["group"] = sub.obs[celltype_col].astype(str)

    sc.tl.rank_genes_groups(
        sub,
        groupby="group",
        groups=[group_a],
        reference=group_b,
        method="wilcoxon",
        corr_method="benjamini-hochberg",
        pts=True,
    )

    result = sc.get.rank_genes_groups_df(sub, group=group_a)
    result = result.rename(columns={
        "names"       : "gene",
        "logfoldchanges": "log2FC",
        "pvals_adj"   : "padj",
        "pvals"       : "pval",
        "scores"      : "score",
    })

    result["-log10padj"] = -np.log10(result["padj"].clip(lower=1e-300))
    result["comparison"] = f"{group_a}_vs_LayerVET"

    sig = result[
        (result["padj"] < pval_thresh) &
        (result["log2FC"].abs() >= log2fc_thresh)
    ].copy()

    sig["direction"] = np.where(sig["log2FC"] > 0, "UP_in_other", "UP_in_LayerVET")

    return result, sig


all_comparisons   = {}   # full results
all_sig           = {}   # significant only

all_cell_types = NON_NEURON_TYPES + UPPER_LAYER_TYPES + OTHER_DEEP_LAYER_TYPES + INTERNEURON_TYPES

for ct in all_cell_types:
    if ct not in adata_use.obs[CELLTYPE_COL].values:
        print(f"[SKIP] '{ct}' not found in data")
        continue
    print(f"[DE]  {ct} vs {LAYER_V_ET} ...", end=" ")
    full, sig = run_de(adata_use, CELLTYPE_COL, ct, LAYER_V_ET, LOG2FC_THRESH, PVAL_THRESH)
    all_comparisons[ct] = full
    all_sig[ct]         = sig
    print(f"{len(sig)} significant genes (UP_other={( sig.direction=='UP_in_other').sum()}, UP_LayerVET={(sig.direction=='UP_in_LayerVET').sum()})")

[DE]  Astro vs L5 ET ... 5330 significant genes (UP_other=943, UP_LayerVET=4387)
[DE]  Endo vs L5 ET ... 5527 significant genes (UP_other=1166, UP_LayerVET=4361)
[DE]  SMC vs L5 ET ... 3012 significant genes (UP_other=573, UP_LayerVET=2439)
[DE]  VLMC vs L5 ET ... 4978 significant genes (UP_other=586, UP_LayerVET=4392)
[DE]  L2/3 IT vs L5 ET ... 1837 significant genes (UP_other=1170, UP_LayerVET=667)
[DE]  L5 IT vs L5 ET ... 1541 significant genes (UP_other=875, UP_LayerVET=666)
[DE]  L5/6 NP vs L5 ET ... 1839 significant genes (UP_other=911, UP_LayerVET=928)
[DE]  L6 CT vs L5 ET ... 1483 significant genes (UP_other=782, UP_LayerVET=701)
[DE]  L6 IT vs L5 ET ... 1707 significant genes (UP_other=1039, UP_LayerVET=668)
[DE]  L6 IT Car3 vs L5 ET ... 1558 significant genes (UP_other=905, UP_LayerVET=653)
[DE]  L6b vs L5 ET ... 1645 significant genes (UP_other=899, UP_LayerVET=746)
[DE]  Lamp5 vs L5 ET ... 2546 significant genes (UP_other=1412, UP_LayerVET=1134)
[DE]  Pvalb vs L5 ET ... 246

## 6. Save Individual Comparison CSVs

In [8]:
for ct, sig in all_sig.items():
    safe_name = ct.replace("/", "_").replace(" ", "_")
    path = OUT_DIR / f"{safe_name}_vs_LayerVET.csv"
    sig.sort_values("log2FC", ascending=False).to_csv(path, index=False)
    print(f"Saved: {path.name}")

Saved: Astro_vs_LayerVET.csv
Saved: Endo_vs_LayerVET.csv
Saved: SMC_vs_LayerVET.csv
Saved: VLMC_vs_LayerVET.csv
Saved: L2_3_IT_vs_LayerVET.csv
Saved: L5_IT_vs_LayerVET.csv
Saved: L5_6_NP_vs_LayerVET.csv
Saved: L6_CT_vs_LayerVET.csv
Saved: L6_IT_vs_LayerVET.csv
Saved: L6_IT_Car3_vs_LayerVET.csv
Saved: L6b_vs_LayerVET.csv
Saved: Lamp5_vs_LayerVET.csv
Saved: Pvalb_vs_LayerVET.csv
Saved: Sncg_vs_LayerVET.csv
Saved: Sst_vs_LayerVET.csv
Saved: Vip_vs_LayerVET.csv


## 7. Stepwise Candidate Narrowing

Logic:
- **Neuron-identity genes** = UP in Layer V ET vs ANY non-neuron type
- **Exitory-Neuron-identity genes** = above AND UP in Layer V ET vs ANY interneurons type
- **Deep-layer-identity genes** = above AND UP in Layer V ET vs ANY upper layer type
- **Layer V ET-specific genes** = above AND UP in Layer V ET vs ALL other deep layer types

In [9]:
def get_layerVET_up_genes(all_sig, cell_types):
    """Genes consistently UP in Layer V ET (direction == UP_in_LayerVET) across given comparisons."""
    sets = []
    for ct in cell_types:
        if ct in all_sig:
            up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
            sets.append(up)
    if not sets:
        return set()
    # Gene must appear in AT LEAST ONE comparison to be included at each step
    return set.union(*sets)


# Step 1: Neuron identity — UP in Layer V ET vs non-neurons
neuron_identity_genes = get_layerVET_up_genes(all_sig, NON_NEURON_TYPES)
print(f"Step 1 — Neuron-identity genes (UP vs non-neurons):  {len(neuron_identity_genes)}")

# Step 2: Exitory Neuron identity — above AND UP vs interneuron types
interneuron_up = get_layerVET_up_genes(all_sig, INTERNEURON_TYPES)
exitory_neuron_identity_genes = neuron_identity_genes & interneuron_up
print(f"Step 2 — Exitory-Neuron-identity genes (also UP vs interneurons):  {len(exitory_neuron_identity_genes)}")

# Step 3: Deep layer identity — above AND UP vs upper layer types
upper_layer_up = get_layerVET_up_genes(all_sig, UPPER_LAYER_TYPES)
deep_layer_identity_genes = exitory_neuron_identity_genes & upper_layer_up
print(f"Step 3 — Deep-layer-identity genes (also UP vs upper layer): {len(deep_layer_identity_genes)}")

# Step 4: Layer V ET specific — above AND UP vs ALL other deep layer types
# Must be UP in Layer V ET in EVERY other-deep-layer comparison (intersection)
deep_layer_sets = []
for ct in OTHER_DEEP_LAYER_TYPES:
    if ct in all_sig:
        up = set(all_sig[ct][all_sig[ct]["direction"] == "UP_in_LayerVET"]["gene"])
        deep_layer_sets.append(up)

if deep_layer_sets:
    consistently_up_vs_deep = set.intersection(*deep_layer_sets)
    layerVET_specific_genes = deep_layer_identity_genes & consistently_up_vs_deep
else:
    layerVET_specific_genes = set()

print(f"Step 3 — Layer V ET-specific genes (UP vs ALL other deep layer): {len(layerVET_specific_genes)}")

Step 1 — Neuron-identity genes (UP vs non-neurons):  6765
Step 2 — Exitory-Neuron-identity genes (also UP vs interneurons):  1870
Step 3 — Deep-layer-identity genes (also UP vs upper layer): 418
Step 3 — Layer V ET-specific genes (UP vs ALL other deep layer): 65


## 8. Build Summary Tables & Save

In [10]:
# ── Gene symbol mapping ───────────────────────────────────────────────────────
id_to_symbol = adata.var['feature_name'].to_dict()
print(f"Loaded {len(id_to_symbol)} gene symbols — example: {list(id_to_symbol.items())[:3]}")

# ── Collect stats for each candidate gene across all comparisons ──────────────
def build_summary(gene_set, all_comparisons, label):
    rows = []
    for gene in sorted(gene_set):
        row = {
            "gene":        gene,
            "gene_symbol": id_to_symbol.get(gene, gene),  # fallback to ID if not found
            "category":    label,
        }
        for ct, df in all_comparisons.items():
            match = df[df["gene"] == gene]
            if not match.empty:
                row[f"{ct}__log2FC"]    = round(match["log2FC"].values[0], 3)
                row[f"{ct}__log10padj"] = round(match["-log10padj"].values[0], 3)
                row[f"{ct}__padj"]      = match["padj"].values[0]
        rows.append(row)
    return pd.DataFrame(rows)

df_neuron   = build_summary(neuron_identity_genes,          all_comparisons, "neuron_identity")
df_exitory  = build_summary(exitory_neuron_identity_genes,  all_comparisons, "exitory_neuron_identity")
df_deep     = build_summary(deep_layer_identity_genes,      all_comparisons, "deep_layer_identity")
df_specific = build_summary(layerVET_specific_genes,        all_comparisons, "LayerVET_specific")

# ── Save ──────────────────────────────────────────────────────────────────────
df_neuron.to_csv(  OUT_DIR / "neuron_identity_genes.csv",          index=False)
df_exitory.to_csv( OUT_DIR / "exitory_neuron_identity_genes.csv",  index=False)
df_deep.to_csv(    OUT_DIR / "deep_layer_identity_genes.csv",      index=False)
df_specific.to_csv(OUT_DIR / "LayerVET_specific_genes.csv",        index=False)

print(f"Saved to {OUT_DIR}/")
print(f"  neuron_identity_genes.csv              : {len(df_neuron)} genes")
print(f"  exitory_neuron_identity_genes.csv      : {len(df_exitory)} genes")
print(f"  deep_layer_identity_genes.csv          : {len(df_deep)} genes")
print(f"  LayerVET_specific_genes.csv            : {len(df_specific)} genes")

Loaded 30198 gene symbols — example: [('ENSMUSG00000029422', 'Rsrc2'), ('ENSMUSG00000114536', 'Gm48837'), ('ENSMUSG00000049036', 'Tmem121')]
Saved to /home/nakagawa/datasets/LayerV_ET_results_rnaseq/
  neuron_identity_genes.csv              : 6765 genes
  exitory_neuron_identity_genes.csv      : 1870 genes
  deep_layer_identity_genes.csv          : 418 genes
  LayerVET_specific_genes.csv            : 65 genes


## 9. Preview Results

In [11]:
print("\n=== TOP 20 Layer V ET-SPECIFIC GENES ===")
if not df_specific.empty:
    log2fc_cols = [c for c in df_specific.columns if c.endswith("__log2FC")]
    df_specific["mean_log2FC"] = df_specific[log2fc_cols].mean(axis=1)
    display(df_specific[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))
else:
    print("No Layer V ET-specific genes found — consider relaxing thresholds (LOG2FC_THRESH / PVAL_THRESH)")

print("\n=== TOP 20 DEEP LAYER IDENTITY GENES ===")
if not df_deep.empty:
    log2fc_cols = [c for c in df_deep.columns if c.endswith("__log2FC")]
    df_deep["mean_log2FC"] = df_deep[log2fc_cols].mean(axis=1)
    display(df_deep[["gene", "gene_symbol", "mean_log2FC"] + log2fc_cols]  # ← added gene_symbol
            .sort_values("mean_log2FC", ascending=False)
            .head(20)
            .reset_index(drop=True))


=== TOP 20 Layer V ET-SPECIFIC GENES ===


,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,SMC__log2FC,VLMC__log2FC,L2/3 IT__log2FC,L5 IT__log2FC,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6 IT Car3__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000054920,Klhl5,-0.971875,0.870,2.554,-1.820,-0.171,-1.503,-1.492,-2.346,-2.512,-1.361,-1.896,-2.546,-2.117,-0.468,-0.017,-0.306,-0.419
1,ENSMUSG00000031451,Gas6,-1.373375,-1.619,0.127,0.340,-4.846,-2.022,-1.702,-1.692,-1.550,-1.690,-1.290,-1.795,-1.188,-0.584,-0.712,-0.771,-0.980
2,ENSMUSG00000064360,mt-Nd3,-1.413750,-0.688,-0.939,0.086,-1.374,-1.895,-2.104,-2.514,-1.555,-1.633,-1.038,-1.790,-1.830,-0.377,-1.405,-1.604,-1.960
3,ENSMUSG00000037110,Ralgapa2,-1.594875,-3.726,-3.025,0.318,-2.897,-1.445,-1.539,-1.139,-1.362,-1.195,-1.998,-1.663,-1.288,-0.745,-1.396,-1.152,-1.266
4,ENSMUSG00000024887,Asah2,-1.788500,-2.985,-1.935,-0.247,-1.021,-2.414,-2.820,-1.248,-1.860,-2.209,-3.688,-1.835,-1.049,0.034,-2.165,-1.476,-1.698
5,ENSMUSG00000028995,Hycc1,-1.861250,-1.319,0.040,-2.311,-3.542,-1.321,-3.769,-1.296,-3.500,-2.484,-2.108,-2.407,-0.444,-1.411,-1.524,-0.852,-1.532
6,ENSMUSG00000038718,Pbx3,-2.010688,-2.348,0.250,-1.078,-0.197,-2.302,-3.341,-3.393,-3.964,-3.192,-2.992,-3.704,-0.765,-2.818,1.077,-3.402,-0.002
7,ENSMUSG00000004558,Ndrg2,-2.054500,4.000,-5.541,1.987,0.780,-6.196,-4.043,-1.445,-1.918,-6.430,-3.765,-2.838,-3.580,-0.109,-1.307,-1.488,-0.979
8,ENSMUSG00000029167,Ppargc1a,-2.108125,-0.500,-5.010,-4.871,-4.879,-1.274,-2.772,-1.095,-2.294,-1.759,-3.404,-1.458,-1.918,0.920,-2.079,-0.486,-0.851
9,ENSMUSG00000027748,Trpc4,-2.160125,-4.955,-5.561,-2.192,0.877,-1.941,-2.448,-4.241,-4.029,-1.717,-1.244,-4.474,-1.912,0.364,0.248,-1.357,0.020



=== TOP 20 DEEP LAYER IDENTITY GENES ===


,gene,gene_symbol,mean_log2FC,Astro__log2FC,Endo__log2FC,SMC__log2FC,VLMC__log2FC,L2/3 IT__log2FC,L5 IT__log2FC,L5/6 NP__log2FC,L6 CT__log2FC,L6 IT__log2FC,L6 IT Car3__log2FC,L6b__log2FC,Lamp5__log2FC,Pvalb__log2FC,Sncg__log2FC,Sst__log2FC,Vip__log2FC
0,ENSMUSG00000047881,Rell1,0.350750,-1.112,2.254,2.164,1.905,-1.939,0.179,2.647,2.718,-2.296,-0.248,2.030,-1.526,-0.701,0.908,-0.633,-0.738
1,ENSMUSG00000022358,Fbxo32,-0.173437,-1.378,-0.334,2.520,-0.802,-2.496,1.437,-2.063,1.100,1.299,1.117,0.395,1.997,-0.655,-1.542,-0.828,-2.542
2,ENSMUSG00000039234,Sec24d,-0.193250,-1.146,0.294,1.823,-0.158,-2.264,-1.279,1.920,-0.233,-0.441,-1.444,1.588,0.731,-2.063,0.223,-0.322,-0.321
3,ENSMUSG00000020728,Cep112,-0.231813,-1.711,0.174,0.112,0.815,-1.159,-0.692,0.670,1.061,-0.714,-2.504,1.583,1.162,-0.461,-1.770,0.293,-0.568
4,ENSMUSG00000059203,Il1rapl2,-0.348500,-1.568,-1.548,-0.276,-2.073,-1.723,1.843,-1.659,-1.690,2.123,0.548,-1.716,-1.458,2.913,-1.199,2.303,-0.396
5,ENSMUSG00000078453,Abracl,-0.404500,-2.674,-0.367,0.949,-0.490,-1.223,1.181,0.780,1.301,0.254,-3.231,1.311,-0.822,-1.325,-0.588,-0.462,-1.066
6,ENSMUSG00000030352,Tspan9,-0.412125,2.560,2.581,0.715,-1.126,-2.283,0.469,1.566,-4.320,-3.803,1.864,-1.553,-0.142,1.975,-2.236,-2.221,-0.640
7,ENSMUSG00000093930,Hmgcs1,-0.440250,0.325,-1.622,-2.428,-1.904,-1.120,-0.112,0.271,0.152,0.504,1.002,0.854,-1.057,-0.688,-0.434,-0.250,-0.537
8,ENSMUSG00000040372,Gpr63,-0.468063,-1.127,-2.392,-1.924,-2.627,-1.254,-0.261,2.040,-0.155,-1.092,-0.475,0.542,2.395,0.139,0.326,-1.080,-0.544
9,ENSMUSG00000021585,Cast,-0.482875,-3.687,3.111,3.087,1.060,-2.246,-0.574,-0.847,-0.802,-2.195,1.127,-1.277,-2.042,-0.596,-1.236,0.455,-1.064


## 10. Threshold Sensitivity Check (Optional)
Run this if Step 3 returns too few or too many genes.

In [12]:
print(adata.var.columns.tolist())
print(adata.var.head())

['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']
                    feature_is_filtered feature_name feature_reference  \
ENSMUSG00000029422                False        Rsrc2   NCBITaxon:10090   
ENSMUSG00000114536                False      Gm48837   NCBITaxon:10090   
ENSMUSG00000049036                False      Tmem121   NCBITaxon:10090   
ENSMUSG00000029577                False        Ube3b   NCBITaxon:10090   
ENSMUSG00000040746                False       Rnf167   NCBITaxon:10090   

                   feature_biotype feature_length    feature_type  
ENSMUSG00000029422            gene           1377  protein_coding  
ENSMUSG00000114536            gene           2849          lncRNA  
ENSMUSG00000049036            gene           1574  protein_coding  
ENSMUSG00000029577            gene           3879  protein_coding  
ENSMUSG00000040746            gene            867  protein_coding  


In [16]:
print("Sensitivity check — gene counts at different thresholds:\n")
print(f"{'log2FC':>8}  {'padj':>6}  {'Neuron':>8}  {'DeepLayer':>10}  {'LayerVET_specific':>18}")

for lfc in [0.5, 1.0, 1.5, 2.0]:
    for pv in [0.1, 0.05, 0.01]:
        def get_up(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.union(*sets) if sets else set()

        def get_up_intersect(cts):
            sets = []
            for ct in cts:
                if ct in all_comparisons:
                    df = all_comparisons[ct]
                    up = set(df[(df["padj"] < pv) & (df["log2FC"] < -lfc)]["gene"])
                    sets.append(up)
            return set.intersection(*sets) if sets else set()

        n  = len(get_up(NON_NEURON_TYPES))
        en = len(get_up(NON_NEURON_TYPES) & get_up(INTERNEURON_TYPES))
        ul = len(get_up(NON_NEURON_TYPES) & get_up(INTERNEURON_TYPES) & get_up(UPPER_LAYER_TYPES))
        sp = len(get_up(NON_NEURON_TYPES) & get_up(UPPER_LAYER_TYPES) & get_up_intersect(OTHER_DEEP_LAYER_TYPES))
        print(f"{lfc:>8.1f}  {pv:>6.2f}  {n:>8}  {ul:>10}  {sp:>18}")

Sensitivity check — gene counts at different thresholds:

  log2FC    padj    Neuron   DeepLayer   LayerVET_specific
     0.5    0.10      8630         890                 177
     0.5    0.05      8375         845                 167
     0.5    0.01      7882         762                 139
     1.0    0.10      7039         435                  73
     1.0    0.05      6765         418                  71
     1.0    0.01      6271         387                  61
     1.5    0.10      5677         268                  39
     1.5    0.05      5387         259                  37
     1.5    0.01      4856         237                  31
     2.0    0.10      4604         174                  28
     2.0    0.05      4373         171                  26
     2.0    0.01      3911         157                  21
